<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model04_Bureau_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [3]:
application = pd.read_csv(DATA_PATH + "application_train.csv")
prev = pd.read_csv(DATA_PATH + "previous_application.csv")
bureau = pd.read_csv(DATA_PATH + "bureau.csv")

print("Application shape:", application.shape)
print("Previous application shape:", prev.shape)
print("Bureau shape:", bureau.shape)

Application shape: (307511, 122)
Previous application shape: (1670214, 37)
Bureau shape: (1716428, 17)


In [4]:
#MODEL02 application features

# DAYS_EMPLOYED anomaly
application["DAYS_EMPLOYED_ANOM"] = (application["DAYS_EMPLOYED"] == 365243).astype(int)
application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

# EXT_SOURCE missing indicators
application["EXT_SOURCE_1_MISSING"] = application["EXT_SOURCE_1"].isna().astype(int)
application["EXT_SOURCE_3_MISSING"] = application["EXT_SOURCE_3"].isna().astype(int)

# Age
application["AGE_YEARS"] = -application["DAYS_BIRTH"] / 365.25

# Employment
application["EMPLOYMENT_YEARS"] = -application["DAYS_EMPLOYED"] / 365.25

# Financial ratios
application["CREDIT_INCOME_RATIO"] = application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_INCOME_RATIO"] = application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_CREDIT_RATIO"] = application["AMT_ANNUITY"] / application["AMT_CREDIT"]
application["GOODS_CREDIT_RATIO"] = application["AMT_GOODS_PRICE"] / application["AMT_CREDIT"]

# Employment / Age
application["EMPLOYMENT_AGE_RATIO"] = application["EMPLOYMENT_YEARS"] / application["AGE_YEARS"]

# EXT_SOURCE summary
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
application["EXT_SOURCE_MEAN"] = application[ext_cols].mean(axis=1)
application["EXT_SOURCE_MIN"] = application[ext_cols].min(axis=1)
application["EXT_SOURCE_MAX"] = application[ext_cols].max(axis=1)
application["EXT_SOURCE_STD"] = application[ext_cols].std(axis=1)

# EXT_SOURCE interactions
application["EXT_SOURCE_1_2"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_2"]
application["EXT_SOURCE_1_3"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_3"]
application["EXT_SOURCE_2_3"] = application["EXT_SOURCE_2"] * application["EXT_SOURCE_3"]

print("MODEL02 application features recreated.")
print("Application shape:", application.shape)

MODEL02 application features recreated.
Application shape: (307511, 139)


In [5]:
#MODEL03 previous application features
# History count
prev_count = (
    prev.groupby("SK_ID_CURR")
    .size()
    .rename("PREV_APPLICATION_COUNT")
    .reset_index()
)

# Financial aggregations
financial_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT": ["mean", "max", "sum"],
        "AMT_APPLICATION": ["mean", "max", "sum"],
        "AMT_ANNUITY": ["mean", "max", "sum"],
        "AMT_GOODS_PRICE": ["mean", "max", "sum"],
        "AMT_DOWN_PAYMENT": ["mean", "max"],
        "RATE_DOWN_PAYMENT": ["mean", "max"]
    })
)
financial_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in financial_agg.columns]
financial_agg = financial_agg.reset_index()

# Credit / application relationship
prev["PREV_CREDIT_APPL_RATIO"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)
prev["PREV_CREDIT_APPL_DIFF"] = prev["AMT_CREDIT"] - prev["AMT_APPLICATION"]

relationship_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_CREDIT_APPL_RATIO": ["mean", "max"],
        "PREV_CREDIT_APPL_DIFF": ["mean", "max"]
    })
)
relationship_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in relationship_agg.columns]
relationship_agg = relationship_agg.reset_index()

# Contract status
prev["PREV_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
prev["PREV_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev["PREV_CANCELED"] = (prev["NAME_CONTRACT_STATUS"] == "Canceled").astype(int)
prev["PREV_UNUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Unused offer").astype(int)

status_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_APPROVED": "sum",
        "PREV_REFUSED": "sum",
        "PREV_CANCELED": "sum",
        "PREV_UNUSED": "sum"
    })
    .reset_index()
)
status_agg = status_agg.rename(columns={
    "PREV_APPROVED": "PREV_APPROVED_COUNT",
    "PREV_REFUSED": "PREV_REFUSED_COUNT",
    "PREV_CANCELED": "PREV_CANCELED_COUNT",
    "PREV_UNUSED": "PREV_UNUSED_COUNT"
})

status_agg = status_agg.merge(
    prev_count[["SK_ID_CURR", "PREV_APPLICATION_COUNT"]],
    on="SK_ID_CURR", how="left"
)
status_agg["PREV_APPROVAL_RATE"] = status_agg["PREV_APPROVED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_REFUSAL_RATE"] = status_agg["PREV_REFUSED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_CANCELLATION_RATE"] = status_agg["PREV_CANCELED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg = status_agg.drop(columns=["PREV_APPLICATION_COUNT"])

# Decision timing
decision_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"DAYS_DECISION": ["min", "max", "mean"]})
)
decision_agg.columns = ["PREV_DAYS_DECISION_" + col[1].upper() for col in decision_agg.columns]
decision_agg = decision_agg.reset_index()

# Payment structure
payment_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"CNT_PAYMENT": ["mean", "max", "sum"]})
)
payment_agg.columns = ["PREV_CNT_PAYMENT_" + col[1].upper() for col in payment_agg.columns]
payment_agg = payment_agg.reset_index()

# Merge all previous-application blocks
prev_features = prev_count.copy()
prev_features = prev_features.merge(financial_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(relationship_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(status_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(decision_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(payment_agg, on="SK_ID_CURR", how="left")

print("Previous-application feature table:", prev_features.shape)
print("One row per applicant:", len(prev_features) == prev_features["SK_ID_CURR"].nunique())

Previous-application feature table: (338857, 35)
One row per applicant: True


In [6]:
bureau_days_cols = [c for c in ["DAYS_CREDIT", "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE"] if c in bureau.columns]

for col in bureau_days_cols:
    sentinel_count = (bureau[col] == 365243).sum()
    print(f"{col} - 365243 count:", sentinel_count)

DAYS_CREDIT - 365243 count: 0
DAYS_CREDIT_ENDDATE - 365243 count: 0
DAYS_ENDDATE_FACT - 365243 count: 0
DAYS_CREDIT_UPDATE - 365243 count: 0


In [7]:
bureau_count = (
    bureau.groupby("SK_ID_CURR")
    .size()
    .rename("BUREAU_CREDIT_COUNT")
    .reset_index()
)

display(bureau_count.head())

,SK_ID_CURR,BUREAU_CREDIT_COUNT
0,100001,7
1,100002,8
2,100003,4
3,100004,2
4,100005,3


In [8]:
bureau_financial_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT_SUM": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_LIMIT": ["mean", "max"],
        "AMT_ANNUITY": ["mean"]
    })
)
bureau_financial_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_financial_agg.columns]
bureau_financial_agg = bureau_financial_agg.reset_index()

display(bureau_financial_agg.head())

,SK_ID_CURR,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,BUREAU_AMT_CREDIT_SUM_LIMIT_MAX,BUREAU_AMT_ANNUITY_MEAN
0,100001,207623.571429,378000.0,1453365.000,85240.928571,373239.0,596686.5,0.00000,0.000,3545.357143
1,100002,108131.945625,450000.0,865055.565,49156.200000,245781.0,245781.0,7997.14125,31988.565,0.000000
2,100003,254350.125000,810000.0,1017400.500,0.000000,0.0,0.0,202500.00000,810000.000,NaN
3,100004,94518.900000,94537.8,189037.800,0.000000,0.0,0.0,0.00000,0.000,NaN
4,100005,219042.000000,568800.0,657126.000,189469.500000,543087.0,568408.5,0.00000,0.000,1420.500000


In [9]:
bureau["BUREAU_OVERDUE_FLAG"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)

bureau_overdue_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "CREDIT_DAY_OVERDUE": ["max"],
        "BUREAU_OVERDUE_FLAG": ["sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["max", "sum"]
    })
)

# Fixed: don't double-prepend "BUREAU_" to columns that already have it
bureau_overdue_agg.columns = [
    col[0] + "_" + col[1].upper() if col[0].startswith("BUREAU_")
    else "BUREAU_" + col[0] + "_" + col[1].upper()
    for col in bureau_overdue_agg.columns
]
bureau_overdue_agg = bureau_overdue_agg.reset_index()

display(bureau_overdue_agg.head())
print("Columns:", bureau_overdue_agg.columns.tolist())

,SK_ID_CURR,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_OVERDUE_FLAG_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM
0,100001,0,0,0.0,0.0
1,100002,0,0,0.0,0.0
2,100003,0,0,0.0,0.0
3,100004,0,0,0.0,0.0
4,100005,0,0,0.0,0.0


Columns: ['SK_ID_CURR', 'BUREAU_CREDIT_DAY_OVERDUE_MAX', 'BUREAU_OVERDUE_FLAG_SUM', 'BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX', 'BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM']


In [10]:
bureau_overdue_agg = bureau_overdue_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)

bureau_overdue_agg["BUREAU_OVERDUE_RATIO"] = (
    bureau_overdue_agg["BUREAU_OVERDUE_FLAG_SUM"] / bureau_overdue_agg["BUREAU_CREDIT_COUNT"]
)

bureau_overdue_agg = bureau_overdue_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

display(bureau_overdue_agg.head())

,SK_ID_CURR,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_OVERDUE_FLAG_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_OVERDUE_RATIO
0,100001,0,0,0.0,0.0,0.0
1,100002,0,0,0.0,0.0,0.0
2,100003,0,0,0.0,0.0,0.0
3,100004,0,0,0.0,0.0,0.0
4,100005,0,0,0.0,0.0,0.0


In [11]:
bureau["BUREAU_ACTIVE_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau["BUREAU_CLOSED_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Closed").astype(int)
bureau["BUREAU_SOLD_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Sold").astype(int)
bureau["BUREAU_BAD_DEBT_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Bad debt").astype(int)

bureau_status_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_ACTIVE_FLAG": "sum",
        "BUREAU_CLOSED_FLAG": "sum",
        "BUREAU_SOLD_FLAG": "sum",
        "BUREAU_BAD_DEBT_FLAG": "sum"
    })
    .reset_index()
)

bureau_status_agg = bureau_status_agg.rename(columns={
    "BUREAU_ACTIVE_FLAG": "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_FLAG": "BUREAU_CLOSED_COUNT",
    "BUREAU_SOLD_FLAG": "BUREAU_SOLD_COUNT",
    "BUREAU_BAD_DEBT_FLAG": "BUREAU_BAD_DEBT_COUNT"
})

display(bureau_status_agg.head())

,SK_ID_CURR,BUREAU_ACTIVE_COUNT,BUREAU_CLOSED_COUNT,BUREAU_SOLD_COUNT,BUREAU_BAD_DEBT_COUNT
0,100001,3,4,0,0
1,100002,2,6,0,0
2,100003,1,3,0,0
3,100004,0,2,0,0
4,100005,2,1,0,0


In [12]:
bureau_status_agg = bureau_status_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)

bureau_status_agg["BUREAU_ACTIVE_RATIO"] = (
    bureau_status_agg["BUREAU_ACTIVE_COUNT"] / bureau_status_agg["BUREAU_CREDIT_COUNT"]
)

bureau_status_agg = bureau_status_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

display(bureau_status_agg.head())

,SK_ID_CURR,BUREAU_ACTIVE_COUNT,BUREAU_CLOSED_COUNT,BUREAU_SOLD_COUNT,BUREAU_BAD_DEBT_COUNT,BUREAU_ACTIVE_RATIO
0,100001,3,4,0,0,0.428571
1,100002,2,6,0,0,0.250000
2,100003,1,3,0,0,0.250000
3,100004,0,2,0,0,0.000000
4,100005,2,1,0,0,0.666667


In [13]:
bureau_type_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CREDIT_TYPE": "nunique"})
    .reset_index()
)
bureau_type_agg = bureau_type_agg.rename(columns={"CREDIT_TYPE": "BUREAU_CREDIT_TYPE_COUNT"})

# Selective important credit types
bureau["BUREAU_CREDIT_CARD_FLAG"] = (bureau["CREDIT_TYPE"] == "Credit card").astype(int)
bureau["BUREAU_MORTGAGE_FLAG"] = (bureau["CREDIT_TYPE"] == "Mortgage").astype(int)
bureau["BUREAU_MICROLOAN_FLAG"] = (bureau["CREDIT_TYPE"] == "Microloan").astype(int)

bureau_type_specific = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_CREDIT_CARD_FLAG": "sum",
        "BUREAU_MORTGAGE_FLAG": "sum",
        "BUREAU_MICROLOAN_FLAG": "sum"
    })
    .reset_index()
)
bureau_type_specific = bureau_type_specific.rename(columns={
    "BUREAU_CREDIT_CARD_FLAG": "BUREAU_CREDIT_CARD_COUNT",
    "BUREAU_MORTGAGE_FLAG": "BUREAU_MORTGAGE_COUNT",
    "BUREAU_MICROLOAN_FLAG": "BUREAU_MICROLOAN_COUNT"
})

bureau_type_agg = bureau_type_agg.merge(bureau_type_specific, on="SK_ID_CURR", how="left")

display(bureau_type_agg.head())

,SK_ID_CURR,BUREAU_CREDIT_TYPE_COUNT,BUREAU_CREDIT_CARD_COUNT,BUREAU_MORTGAGE_COUNT,BUREAU_MICROLOAN_COUNT
0,100001,1,0,0,0
1,100002,2,4,0,0
2,100003,2,2,0,0
3,100004,1,0,0,0
4,100005,2,1,0,0


In [14]:
bureau_timing_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "DAYS_CREDIT": ["min", "max", "mean"],
        "DAYS_CREDIT_UPDATE": ["mean"]
    })
)
bureau_timing_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_timing_agg.columns]
bureau_timing_agg = bureau_timing_agg.reset_index()

display(bureau_timing_agg.head())

,SK_ID_CURR,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MEAN
0,100001,-1572,-49,-735.000000,-93.142857
1,100002,-1437,-103,-874.000000,-499.875000
2,100003,-2586,-606,-1400.750000,-816.000000
3,100004,-1326,-408,-867.000000,-532.000000
4,100005,-373,-62,-190.666667,-54.333333


In [15]:
bureau_prolong_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CNT_CREDIT_PROLONG": "sum"})
    .reset_index()
)
bureau_prolong_agg = bureau_prolong_agg.rename(columns={"CNT_CREDIT_PROLONG": "BUREAU_CREDIT_PROLONG_TOTAL"})

display(bureau_prolong_agg.head())

,SK_ID_CURR,BUREAU_CREDIT_PROLONG_TOTAL
0,100001,0
1,100002,0
2,100003,0
3,100004,0
4,100005,0


In [16]:
bureau_features = bureau_count.copy()
bureau_features = bureau_features.merge(bureau_financial_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_overdue_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_status_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_type_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_timing_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_prolong_agg, on="SK_ID_CURR", how="left")

print("Bureau feature table shape:", bureau_features.shape)
display(bureau_features.head())

Bureau feature table shape: (305811, 30)


,SK_ID_CURR,BUREAU_CREDIT_COUNT,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,BUREAU_AMT_CREDIT_SUM_LIMIT_MAX,...,BUREAU_ACTIVE_RATIO,BUREAU_CREDIT_TYPE_COUNT,BUREAU_CREDIT_CARD_COUNT,BUREAU_MORTGAGE_COUNT,BUREAU_MICROLOAN_COUNT,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_CREDIT_PROLONG_TOTAL
0,100001,7,207623.571429,378000.0,1453365.000,85240.928571,373239.0,596686.5,0.00000,0.000,...,0.428571,1,0,0,0,-1572,-49,-735.000000,-93.142857,0
1,100002,8,108131.945625,450000.0,865055.565,49156.200000,245781.0,245781.0,7997.14125,31988.565,...,0.250000,2,4,0,0,-1437,-103,-874.000000,-499.875000,0
2,100003,4,254350.125000,810000.0,1017400.500,0.000000,0.0,0.0,202500.00000,810000.000,...,0.250000,2,2,0,0,-2586,-606,-1400.750000,-816.000000,0
3,100004,2,94518.900000,94537.8,189037.800,0.000000,0.0,0.0,0.00000,0.000,...,0.000000,1,0,0,0,-1326,-408,-867.000000,-532.000000,0
4,100005,3,219042.000000,568800.0,657126.000,189469.500000,543087.0,568408.5,0.00000,0.000,...,0.666667,2,1,0,0,-373,-62,-190.666667,-54.333333,0


In [17]:
print("Rows:", len(bureau_features))
print("Unique applicants:", bureau_features["SK_ID_CURR"].nunique())
print("One row per applicant:", len(bureau_features) == bureau_features["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", bureau_features["SK_ID_CURR"].duplicated().sum())

Rows: 305811
Unique applicants: 305811
One row per applicant: True
Duplicate applicant IDs: 0


In [18]:
bureau_feature_cols = [col for col in bureau_features.columns if col != "SK_ID_CURR"]

print("Number of bureau features:", len(bureau_feature_cols))
for col in bureau_feature_cols:
    print("-", col)

Number of bureau features: 29
- BUREAU_CREDIT_COUNT
- BUREAU_AMT_CREDIT_SUM_MEAN
- BUREAU_AMT_CREDIT_SUM_MAX
- BUREAU_AMT_CREDIT_SUM_SUM
- BUREAU_AMT_CREDIT_SUM_DEBT_MEAN
- BUREAU_AMT_CREDIT_SUM_DEBT_MAX
- BUREAU_AMT_CREDIT_SUM_DEBT_SUM
- BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN
- BUREAU_AMT_CREDIT_SUM_LIMIT_MAX
- BUREAU_AMT_ANNUITY_MEAN
- BUREAU_CREDIT_DAY_OVERDUE_MAX
- BUREAU_OVERDUE_FLAG_SUM
- BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX
- BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM
- BUREAU_OVERDUE_RATIO
- BUREAU_ACTIVE_COUNT
- BUREAU_CLOSED_COUNT
- BUREAU_SOLD_COUNT
- BUREAU_BAD_DEBT_COUNT
- BUREAU_ACTIVE_RATIO
- BUREAU_CREDIT_TYPE_COUNT
- BUREAU_CREDIT_CARD_COUNT
- BUREAU_MORTGAGE_COUNT
- BUREAU_MICROLOAN_COUNT
- BUREAU_DAYS_CREDIT_MIN
- BUREAU_DAYS_CREDIT_MAX
- BUREAU_DAYS_CREDIT_MEAN
- BUREAU_DAYS_CREDIT_UPDATE_MEAN
- BUREAU_CREDIT_PROLONG_TOTAL


In [19]:
application_model = application.copy()

application_model = application_model.merge(prev_features, on="SK_ID_CURR", how="left")
print("Shape after MODEL03 features:", application_model.shape)

application_model = application_model.merge(bureau_features, on="SK_ID_CURR", how="left")
print("Shape after Bureau features:", application_model.shape)

Shape after MODEL03 features: (307511, 173)
Shape after Bureau features: (307511, 202)


In [20]:
application_model["HAS_BUREAU_HISTORY"] = application_model["BUREAU_CREDIT_COUNT"].notna().astype(int)

print(application_model["HAS_BUREAU_HISTORY"].value_counts())

HAS_BUREAU_HISTORY
1    263491
0     44020
Name: count, dtype: int64


In [21]:
no_bureau_history = application_model["BUREAU_CREDIT_COUNT"].isna().sum()

print("Applicants without Bureau history:", no_bureau_history)
print("Percentage:", no_bureau_history / len(application_model) * 100)

Applicants without Bureau history: 44020
Percentage: 14.314935075493235


In [22]:
X = application_model.drop(columns=["TARGET", "SK_ID_CURR"])
y = application_model["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 201)
y shape: (307511,)


In [23]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training shape: (246008, 201)
Validation shape: (61503, 201)

Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [24]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 185
Categorical features: 16


In [25]:
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [26]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [27]:
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

In [28]:
print("Training XGBoost with MODEL03 + Bureau features...")
xgb_pipeline.fit(X_train, y_train)
print("Training complete.")

Training XGBoost with MODEL03 + Bureau features...
Training complete.


In [29]:
valid_proba = xgb_pipeline.predict_proba(X_valid)[:, 1]
print("Predictions generated.")

Predictions generated.


In [30]:
roc_auc = roc_auc_score(y_valid, valid_proba)
pr_auc = average_precision_score(y_valid, valid_proba)

print("XGBOOST + MODEL03 + BUREAU FEATURES")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

XGBOOST + MODEL03 + BUREAU FEATURES
ROC-AUC: 0.7776
PR-AUC:  0.2742


In [31]:
MODEL03_ROC_AUC = 0.775428
MODEL03_PR_AUC = 0.265853

roc_change = roc_auc - MODEL03_ROC_AUC
pr_change = pr_auc - MODEL03_PR_AUC

print("IMPROVEMENT OVER MODEL03")
print(f"ROC-AUC change: {roc_change:+.4f}")
print(f"PR-AUC change:  {pr_change:+.4f}")

IMPROVEMENT OVER MODEL03
ROC-AUC change: +0.0022
PR-AUC change:  +0.0083


In [32]:
bureau_result = pd.DataFrame({
    "Experiment": ["XGBoost + Model03 + Bureau Features"],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "ROC-AUC Change": [roc_change],
    "PR-AUC Change": [pr_change]
})

display(bureau_result.style.format({
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}",
    "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change
0,XGBoost + Model03 + Bureau Features,0.7776,0.2742,+0.0022,+0.0083


In [33]:
bureau_result = pd.DataFrame({
    "Experiment": ["XGBoost + Model03 + Bureau Features"],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "ROC-AUC Change": [roc_change],
    "PR-AUC Change": [pr_change]
})

display(bureau_result.style.format({
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}",
    "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change
0,XGBoost + Model03 + Bureau Features,0.7776,0.2742,+0.0022,+0.0083


In [34]:
bureau_result.to_csv(RESULTS_PATH + "bureau_experiment.csv", index=False)
print("Experiment saved to:", RESULTS_PATH + "bureau_experiment.csv")

Experiment saved to: /content/drive/MyDrive/RupeeRisk/bureau_experiment.csv


In [35]:
!pip install mlflow -q
import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [36]:
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")

with mlflow.start_run(run_name="XGBoost_Bureau"):
    mlflow.log_param("stage", "Model04 - Bureau Feature Engineering")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("builds_on", "Model03 application + previous application features")
    mlflow.log_param("n_bureau_features", len(bureau_feature_cols))
    mlflow.log_param("scale_pos_weight", False)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc_change_vs_Model03", roc_change)
    mlflow.log_metric("pr_auc_change_vs_Model03", pr_change)

print("Model04 logged to MLflow.")

Model04 logged to MLflow.


In [37]:
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Bureau,0.777585,0.274161
1,XGBoost_Previous_Application,0.775428,0.265853
2,XGBoost_Application_Features,0.769403,0.262725
3,XGBoost_scale_pos_weight,0.760000,0.249300
4,XGBoost_Baseline,0.761200,0.251600
5,Logistic_Regression_Baseline,0.750100,0.232600


In [38]:
print("Bureau features created:")
for col in bureau_feature_cols:
    print("-", col)
print("\nTotal Bureau features:", len(bureau_feature_cols))

Bureau features created:
- BUREAU_CREDIT_COUNT
- BUREAU_AMT_CREDIT_SUM_MEAN
- BUREAU_AMT_CREDIT_SUM_MAX
- BUREAU_AMT_CREDIT_SUM_SUM
- BUREAU_AMT_CREDIT_SUM_DEBT_MEAN
- BUREAU_AMT_CREDIT_SUM_DEBT_MAX
- BUREAU_AMT_CREDIT_SUM_DEBT_SUM
- BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN
- BUREAU_AMT_CREDIT_SUM_LIMIT_MAX
- BUREAU_AMT_ANNUITY_MEAN
- BUREAU_CREDIT_DAY_OVERDUE_MAX
- BUREAU_OVERDUE_FLAG_SUM
- BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX
- BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM
- BUREAU_OVERDUE_RATIO
- BUREAU_ACTIVE_COUNT
- BUREAU_CLOSED_COUNT
- BUREAU_SOLD_COUNT
- BUREAU_BAD_DEBT_COUNT
- BUREAU_ACTIVE_RATIO
- BUREAU_CREDIT_TYPE_COUNT
- BUREAU_CREDIT_CARD_COUNT
- BUREAU_MORTGAGE_COUNT
- BUREAU_MICROLOAN_COUNT
- BUREAU_DAYS_CREDIT_MIN
- BUREAU_DAYS_CREDIT_MAX
- BUREAU_DAYS_CREDIT_MEAN
- BUREAU_DAYS_CREDIT_UPDATE_MEAN
- BUREAU_CREDIT_PROLONG_TOTAL

Total Bureau features: 29


In [40]:
print("""
Model04 BUREAU FEATURE ENGINEERING COMPLETE

Model03 benchmark: ROC-AUC = 0.775428, PR-AUC = 0.265853
Model04 result:    ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Change:            ROC-AUC = {:+.4f}, PR-AUC = {:+.4f}


Next: bureau_balance, POS_CASH_balance, installments_payments,
      credit_card_balance, then feature selection, Optuna, SHAP.
""".format(roc_auc, pr_auc, roc_change, pr_change))


Model04 BUREAU FEATURE ENGINEERING COMPLETE

Model03 benchmark: ROC-AUC = 0.775428, PR-AUC = 0.265853
Model04 result:    ROC-AUC = 0.7776, PR-AUC = 0.2742
Change:            ROC-AUC = +0.0022, PR-AUC = +0.0083


Next: bureau_balance, POS_CASH_balance, installments_payments,
      credit_card_balance, then feature selection, Optuna, SHAP.

